# CAMELS Scorecard — Greek Systemic Banks (2022–2024)

A CAMELS-inspired 1–5 scoring framework using publicly available KPIs from official annual reports.
Rating scale: **1 = Strong / 5 = Unsatisfactory**, calibrated to European bank supervisory norms (ECB SSM / EBA guidelines).

| Dimension | Proxy Metric | Direction |
|-----------|-------------|----------|
| **C** Capital | CET1 ratio | Higher → better |
| **A** Asset Quality | Cost of Risk = \|Impairment\| / Loans (%) | Lower → better |
| **M** Management | Cost-to-Income ratio (%) | Lower → better |
| **E** Earnings | ROE (%) | Higher → better |
| **L** Liquidity | Loan-to-Deposit ratio (%) | Lower → better |

**Note:** True CAMELS includes a Sensitivity-to-market-risk (S) dimension requiring VaR / trading book data not in our dataset.
Full NPL ratio would strengthen the A dimension — data limitation noted in `DATA_DICTIONARY.md`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
YEARS = [2022, 2023, 2024]

DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
print(f'Loaded {len(kpis)} rows. Columns: {list(kpis.columns)}')

In [ ]:
# ── Derive CAMELS metrics ─────────────────────────────────────────────────────
df = kpis.copy()

# Cost of Risk = |Impairment losses| / Gross Loans × 100
# impairment column is negative, so abs() to get positive basis points / %
df['cost_of_risk'] = (df['impairment'].abs() / df['loans']) * 100

# CAMELS dimensions → raw metric columns
camels_metrics = {
    'C — Capital':       ('cet1',          'higher'),
    'A — Asset Quality': ('cost_of_risk',   'lower'),
    'M — Management':    ('cost_to_income', 'lower'),
    'E — Earnings':      ('roe',            'higher'),
    'L — Liquidity':     ('loan_to_deposit','lower'),
}

print('CAMELS metrics computed:')
print(df[['bank','year','cet1','cost_of_risk','cost_to_income','roe','loan_to_deposit']].round(2).to_string(index=False))

In [ ]:
# ── Scoring thresholds (European banking supervisory calibration) ─────────────
#
# Thresholds sourced from:
# - ECB SSM SREP methodology (CET1)
# - EBA Risk Dashboard (cost of risk, C/I benchmarks)
# - Standard equity research peer analysis

def score_metric(value, thresholds, direction):
    """
    Return 1 (best) to 5 (worst).
    thresholds: list of 4 cut-points dividing the [1,2,3,4,5] regions.
    direction:  'higher' (higher = better) or 'lower' (lower = better).
    """
    if pd.isna(value):
        return np.nan
    t = sorted(thresholds)
    if direction == 'higher':
        if   value >= t[3]: return 1
        elif value >= t[2]: return 2
        elif value >= t[1]: return 3
        elif value >= t[0]: return 4
        else:               return 5
    else:  # lower is better
        if   value <= t[0]: return 1
        elif value <= t[1]: return 2
        elif value <= t[2]: return 3
        elif value <= t[3]: return 4
        else:               return 5

THRESHOLDS = {
    'cet1':           ([11.0, 13.0, 15.0, 18.0], 'higher'),  # %
    'cost_of_risk':   ([0.50, 0.80, 1.20, 1.60], 'lower'),   # %
    'cost_to_income': ([30.0, 37.0, 45.0, 55.0], 'lower'),   # %
    'roe':            ([5.0,  8.0,  12.0, 15.0],  'higher'),  # %
    'loan_to_deposit':([60.0, 65.0, 75.0, 85.0], 'lower'),   # %
}

score_cols = {}
for dim, (metric, _) in camels_metrics.items():
    thresholds, direction = THRESHOLDS[metric]
    col = f'score_{metric}'
    df[col] = df[metric].apply(lambda v: score_metric(v, thresholds, direction))
    score_cols[dim] = col

# Composite CAMELS score (simple average)
score_col_list = list(score_cols.values())
df['camels_composite'] = df[score_col_list].mean(axis=1).round(1)

print('Scores computed:')
display_df = df[['bank','year'] + score_col_list + ['camels_composite']].copy()
print(display_df.to_string(index=False))

In [ ]:
# ── Chart 1: CAMELS Heatmap — 3 panels (one per year) ────────────────────────
# Each cell: colour = score (1=green, 5=red), annotation = raw metric value

colorscale = [
    [0.0,  '#1a7a4a'],  # 1 — Strong (dark green)
    [0.25, '#5cb85c'],  # 2 — Good (light green)
    [0.5,  '#f0ad4e'],  # 3 — Satisfactory (amber)
    [0.75, '#d9534f'],  # 4 — Marginal (red)
    [1.0,  '#7b1a1a'],  # 5 — Unsatisfactory (dark red)
]

dim_short = ['C (CET1)', 'A (CoR)', 'M (C/I)', 'E (ROE)', 'L (LtD)']
metric_fmts = {
    'cet1':           '{:.1f}%',
    'cost_of_risk':   '{:.2f}%',
    'cost_to_income': '{:.1f}%',
    'roe':            '{:.1f}%',
    'loan_to_deposit':'{:.1f}%',
}

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{y}</b>' for y in YEARS],
    horizontal_spacing=0.06,
)

metric_order = ['cet1','cost_of_risk','cost_to_income','roe','loan_to_deposit']

for ci, year in enumerate(YEARS):
    yr_df = df[df['year'] == year].set_index('bank').reindex(BANKS)

    z_matrix = np.array([[yr_df.loc[b, f'score_{m}'] for m in metric_order] for b in BANKS], dtype=float)
    text_matrix = [
        [metric_fmts[m].format(yr_df.loc[b, m]) + f'<br><b>{int(yr_df.loc[b, "score_" + m])}</b>'
         for m in metric_order]
        for b in BANKS
    ]

    fig1.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=dim_short,
            y=BANKS,
            text=text_matrix,
            texttemplate='%{text}',
            colorscale=colorscale,
            zmin=1, zmax=5,
            showscale=(ci == 2),
            colorbar=dict(title='Score', tickvals=[1,2,3,4,5],
                          ticktext=['1 Strong','2 Good','3 OK','4 Marginal','5 Weak'],
                          len=0.6),
            hoverongaps=False,
        ),
        row=1, col=ci + 1,
    )

fig1.update_layout(
    title_text='<b>CAMELS Scorecard</b> | Greek Systemic Banks 2022–2024 | 1 = Strong, 5 = Weak',
    title_font_size=16,
    template='plotly_dark',
    height=400,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
    font=dict(size=12),
)
fig1.show()

In [ ]:
# ── Chart 2: Composite CAMELS Score Trend ─────────────────────────────────────
fig2 = go.Figure()
for bank in BANKS:
    bdf = df[df['bank'] == bank].sort_values('year')
    fig2.add_trace(go.Scatter(
        x=bdf['year'],
        y=bdf['camels_composite'],
        name=bank,
        mode='lines+markers+text',
        line=dict(color=COLORS[bank], width=2.5),
        marker=dict(size=10),
        text=bdf['camels_composite'].astype(str),
        textposition='top center',
    ))

fig2.update_layout(
    title_text='<b>Composite CAMELS Score</b> | Simple average of 5 dimensions | Lower = Stronger',
    xaxis=dict(tickvals=YEARS, ticktext=[str(y) for y in YEARS]),
    yaxis=dict(title='Composite Score (1=Strong, 5=Weak)', range=[0.5, 5.5], autorange='reversed'),
    template='plotly_dark',
    height=380,
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.show()

In [ ]:
# ── Chart 3: 2024 Ranked Summary Table ───────────────────────────────────────
df24 = df[df['year'] == 2024].copy().sort_values('camels_composite')

print('\n── 2024 CAMELS Rankings (sorted by composite score, lower = stronger) ──')
rank_df = df24[['bank'] + [f'score_{m}' for m in metric_order] + ['camels_composite']].copy()
rank_df.columns = ['Bank', 'C (Capital)', 'A (Asset)', 'M (Mgmt)', 'E (Earn)', 'L (Liq)', 'Composite']
rank_df = rank_df.reset_index(drop=True)
rank_df.index = rank_df.index + 1
rank_df.index.name = 'Rank'
print(rank_df.to_string())

print('\n── Threshold Reference ──')
thresh_info = [
    ('C (CET1)',        '> 18%',    '15-18%',  '13-15%',   '11-13%',   '< 11%'),
    ('A (Cost of Risk)','< 0.50%',  '0.5-0.8%','0.8-1.2%', '1.2-1.6%', '> 1.60%'),
    ('M (Cost/Income)', '< 30%',    '30-37%',  '37-45%',   '45-55%',   '> 55%'),
    ('E (ROE)',         '> 15%',    '12-15%',  '8-12%',    '5-8%',     '< 5%'),
    ('L (Loan/Dep.)',   '< 60%',    '60-65%',  '65-75%',   '75-85%',   '> 85%'),
]
header = f'  {"Dimension":<18} {"Score 1":<12} {"Score 2":<12} {"Score 3":<12} {"Score 4":<12} {"Score 5"}'
print(header)
print('  ' + '-' * 80)
for row in thresh_info:
    print(f'  {row[0]:<18} {row[1]:<12} {row[2]:<12} {row[3]:<12} {row[4]:<12} {row[5]}')

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12,                           'Expected 12 bank-year rows.'
assert df['cost_of_risk'].notna().all(),         'Missing cost_of_risk values.'
assert df['camels_composite'].between(1, 5).all(),'Composite score out of [1,5] range.'

for col in score_col_list:
    assert df[col].between(1, 5).all(), f'{col} has values outside [1,5].'

# Verify 2024 ranking makes intuitive sense: NBG should score well on CET1
nbg24_cet1_score = df[(df['bank'] == 'NBG') & (df['year'] == 2024)]['score_cet1'].values[0]
assert nbg24_cet1_score <= 2, f'NBG 2024 CET1 19.1% should score 1-2, got {nbg24_cet1_score}'

print('✅ All checks passed.')
print(f'   2024 sector avg composite score: {df[df["year"]==2024]["camels_composite"].mean():.2f}')
print(f'   Strongest bank (2024): {df24.iloc[0]["bank"]} (composite {df24.iloc[0]["camels_composite"]})')
print(f'   Weakest bank  (2024): {df24.iloc[-1]["bank"]} (composite {df24.iloc[-1]["camels_composite"]})')